# RFS-FIM Fetch

Pull flood-extent tiles from the public S3 bucket `floodmap-sandbox` that overlap a
bounding-box AOI, for a chosen return period, then clip/mosaic to a single GeoTIFF.

### Setup

In [93]:
# run once if needed
# %pip install s3fs geopandas rioxarray rasterio

import math
import s3fs
import geopandas as gpd
import os
import numpy as np
import rioxarray
from pathlib import Path
from rioxarray.merge import merge_arrays

fs = s3fs.S3FileSystem(anon=True)
BUCKET = "floodmap-sandbox"
DEM = "fabdem"
TIF_NAME = "flows_2,5,10,25,50,100.tif"
RETURN_PERIODS = [2, 5, 10, 25, 50, 100]   # band order in the tif

### 1. Accept AOI (bounding box)

Read the `AOI_bbox` shapefile from the project folder. (Only the bbox is used for now.)

In [94]:
AOI_PATH = "Cuba_Moron.shp"
aoi = gpd.read_file(AOI_PATH)
aoi_name = Path(AOI_PATH).stem 

if aoi.crs is None:
    raise ValueError("AOI has no CRS defined.")
if aoi.crs.to_epsg() != 4326:
    aoi = aoi.to_crs(4326)

aoi_bounds = aoi.total_bounds
print("AOI bounds (min_lon, min_lat, max_lon, max_lat):")
print(aoi_bounds)

AOI bounds (min_lon, min_lat, max_lon, max_lat):
[-78.66601907  22.0707101  -78.59011684  22.13628957]


### 2. Accept return-period selection

Pick return periods.

In [95]:
SELECTED_RPS = [2, 5, 10, 25, 50, 100]
assert all(rp in RETURN_PERIODS for rp in SELECTED_RPS), f"Pick from {RETURN_PERIODS}"
print("Selected return periods:", SELECTED_RPS)

Selected return periods: [2, 5, 10, 25, 50, 100]


### 3. Derive the 5 variables

From the bbox: floor the minimums and ceiling the maximums to tile boundary extents.

In [96]:
min_lon = math.floor(aoi_bounds[0])
min_lat = math.floor(aoi_bounds[1])
max_lon = math.ceil(aoi_bounds[2])
max_lat = math.ceil(aoi_bounds[3])

print(f"RETURN_PERIOD = {RETURN_PERIODS}")
print(f"lon: {min_lon} -> {max_lon}")
print(f"lat: {min_lat} -> {max_lat}")

RETURN_PERIOD = [2, 5, 10, 25, 50, 100]
lon: -79 -> -78
lat: 22 -> 23


### 4. Build candidate tile URLs and keep the ones that exist

Loop every integer lon/lat in the AOI range (SW-corner tiles), build the S3 path,
and keep only tiles that actually exist in the bucket. `range(min, max)` is exclusive
on the top end, which is correct for SW-corner 1° tiles.

In [97]:
urls = []
for lon_to_download in range(min_lon, max_lon):
    for lat_to_download in range(min_lat, max_lat):
        key = (f"{BUCKET}/tiles/lon={lon_to_download}/lat={lat_to_download}"
               f"/floodmaps/dem={DEM}/{TIF_NAME}")
        if fs.exists(key):
            urls.append(key)
            print(f"  found: lon={lon_to_download}, lat={lat_to_download}")
        else:
            print(f"  missing: lon={lon_to_download}, lat={lat_to_download}")

print(f"\n{len(urls)} tiles to download.")

  found: lon=-79, lat=22

1 tiles to download.


### 5. Download the tiles locally

Copies each tile into a local `downloads/` folder (bucket is only read).

In [98]:
os.makedirs("downloads", exist_ok=True)
local_paths = []
for url in urls:
    lon = url.split("lon=")[1].split("/")[0]
    lat = url.split("lat=")[1].split("/")[0]
    local = f"downloads/lon{lon}_lat{lat}_{TIF_NAME}"
    if not os.path.exists(local):
        fs.get(url, local)
    local_paths.append(local)
    print("  ", local)

print(f"\n{len(local_paths)} files local.")

   downloads/lon-79_lat22_flows_2,5,10,25,50,100.tif

1 files local.


### 6. Clip + mosaic → result GeoTIFF

- Select the band for the chosen return period.
- Mosaic the tiles together, then clip to the AOI.
- Write the result locally.

In [99]:
arrays = [rioxarray.open_rasterio(p, masked=True).squeeze("band", drop=True)
          for p in local_paths]
mosaic = merge_arrays(arrays) if len(arrays) > 1 else arrays[0]

# value is inverse to return period: 100=2-yr core, 16=100-yr fringe -> descending
vals = sorted((int(v) for v in np.unique(mosaic.values[~np.isnan(mosaic.values)]) if v > 0),
              reverse=True)
rp_to_value = dict(zip(sorted(RETURN_PERIODS), vals))
print("return-period -> pixel-value map:", rp_to_value)

written = []
for rp in SELECTED_RPS:
    thresh = rp_to_value[rp]

    mask = ((mosaic > 0) & (mosaic >= thresh)).astype("uint8")   # 1 = flooded, 0 = dry
    extent = mask.where(mask == 1, 255).astype("uint8")          # dry 0 -> 255 (nodata)
    extent = extent.rio.write_crs(mosaic.rio.crs).rio.write_nodata(255)

    clipped = extent.rio.clip(aoi.geometry.values, aoi.crs, drop=True)
    out_path = f"{aoi_name}_rp{rp}.tif"
    clipped.rio.to_raster(out_path)

    n = int((clipped == 1).sum())
    written.append(out_path)
    print(f"  wrote {out_path}  ({n:,} flooded px in AOI)")

print("\nDone. Files:", written)

return-period -> pixel-value map: {2: 100, 5: 83, 10: 66, 25: 50, 50: 33, 100: 16}
  wrote Cuba_Moron_rp2.tif  (7,823 flooded px in AOI)
  wrote Cuba_Moron_rp5.tif  (9,806 flooded px in AOI)
  wrote Cuba_Moron_rp10.tif  (11,264 flooded px in AOI)
  wrote Cuba_Moron_rp25.tif  (12,934 flooded px in AOI)
  wrote Cuba_Moron_rp50.tif  (14,351 flooded px in AOI)
  wrote Cuba_Moron_rp100.tif  (16,256 flooded px in AOI)

Done. Files: ['Cuba_Moron_rp2.tif', 'Cuba_Moron_rp5.tif', 'Cuba_Moron_rp10.tif', 'Cuba_Moron_rp25.tif', 'Cuba_Moron_rp50.tif', 'Cuba_Moron_rp100.tif']
